# 🚧 Notebook 09 — Putting on Guardrails
### Proximal Policy Optimization (PPO)

**Series**: RL Notebook Series · Act III — Policy-Based Methods · Post 9 of 15

---

## The Danger of Massive Leaps

In Notebook 08 (Actor-Critic), we solved the variance problem using the Advantage function. But we uncovered a new, extremely dangerous problem: **Policy Collapse.**

If the Critic gave the Actor a massive Advantage score, the Actor would take a massive mathematical step during Backpropagation. That huge gradient step would irreversibly scramble the neural network's weights, instantly causing the agent to forget everything it had learned and plummeting its score back down to zero.

Think of a neural network walking blindfolded through a minefield. 
- If it takes slow, 1-inch shuffles, it will eventually find the safe path to the goal.
- If it takes random 10-foot leaps, it *might* survive once, but eventually, it will land on a mine and blow up its own policy.

## Trust Regions & PPO

We need a mathematical guarantee that the Actor is **never allowed to take a massive leap**, regardless of how good the Advantage score is.

We define a **Trust Region**: Before we update our policy, we draw a small boundary around our current state. We promise that our newly updated policy will not step outside that boundary. 

**Proximal Policy Optimization (PPO)** does this using an incredibly elegant trick called the **Clipped Surrogate Objective**. It is the absolute industry standard RL algorithm powering modern AI, from OpenAI's Dota bots to ChatGPT's RLHF tuning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3})

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. The Ratio and the Clip

Instead of updating the probabilities directly with `-log_prob * Advantage` like we did in REINFORCE/A2C, PPO calculates the **Ratio** between the *new* policy we are currently updating, and the *old* policy that originally played the episode.

$$ r_t(\theta) = \frac{\pi_{\text{new}}(a|s)}{\pi_{\text{old}}(a|s)} $$

If $r_t > 1$, the action is more probable now than it was before.

### The Magic Trick
We multiply that ratio by our Advantage... but we use `torch.clamp` to restrict the ratio between `0.8` and `1.2` (an epsilon $\epsilon = 0.2$).

Let's say an action was shockingly good (Positive Advantage). PPO pushes the probability of taking that action up... but once it has increased by `20%` compared to the old policy, **PPO stops rewarding the network.** The gradient hits zero. It restricts the neural network from taking a gigantic leap of faith, completely preventing Policy Collapses.

In [ ]:
class PPOActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(PPOActorCritic, self).__init__()
        # The shared body
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        
        # The Actor Head
        self.actor_fc = nn.Linear(hidden_dim, action_dim)
        
        # The Critic Head
        self.critic_fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, state):
        x = F.relu(self.fc1(state))
        action_logits = self.actor_fc(x)
        action_probs = F.softmax(action_logits, dim=-1)
        state_value = self.critic_fc(x)
        
        return action_probs, state_value
    
    def evaluate(self, state, action):
        """Used during the PPO update loop to get the new log_probs and values"""
        action_probs, state_value = self.forward(state)
        m = Categorical(action_probs)
        # Get the new log probability of the action we took in the past
        log_prob = m.log_prob(action)
        return log_prob, state_value

## 2. Sample Efficiency through Epochs

Because PPO mathematically prevents the policy from moving too far, we can safely reuse the same batch of data multiple times! 

In A2C, we played an episode, did one single backpropagation pass, and threw the data away. 
In PPO, we play an episode, save everything into memory, and then run **Epochs** (e.g., updating our neural network 4 or 5 times using the *exact same experience data*).

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        
    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.log_probs.clear()
        self.rewards.clear()
        self.values.clear()

## 3. The PPO Training Loop

Inside our update function, we will:
1. Convert our RolloutBuffer history into PyTorch Tensors.
2. Calculate the real Returns (just like we did in Notebook 08).
3. Loop `K_epochs` times. In each epoch, calculate the new Ratio.
4. Apply the `torch.clamp` math to restrict the Actor's update.

In [ ]:
def train_ppo(env_name='CartPole-v1', num_episodes=500, gamma=0.99, lr=3e-3, K_epochs=4, eps_clip=0.2):
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    network = PPOActorCritic(state_dim, action_dim)
    optimizer = optim.Adam(network.parameters(), lr=lr)
    memory = RolloutBuffer()
    
    episode_rewards = []
    
    for ep in range(num_episodes):
        state, _ = env.reset()
        done = False
        truncated = False
        ep_reward = 0
        
        # 1. Collect Trajectory (The "Old" Policy)
        while not (done or truncated):
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                action_probs, value = network(state_tensor)
            
            m = Categorical(action_probs)
            action = m.sample()
            log_prob = m.log_prob(action)
            
            next_state, reward, done, truncated, _ = env.step(action.item())
            
            # Store experience in memory
            memory.states.append(state)
            memory.actions.append(action.item())
            memory.log_probs.append(log_prob.item())
            memory.values.append(value.item())
            memory.rewards.append(reward)
            
            state = next_state
            ep_reward += reward
            
        episode_rewards.append(ep_reward)
        
        # 2. Convert memory to tensors
        old_states = torch.FloatTensor(np.array(memory.states))
        old_actions = torch.LongTensor(np.array(memory.actions))
        old_log_probs = torch.FloatTensor(np.array(memory.log_probs))
        old_values = torch.FloatTensor(np.array(memory.values))
        
        # Calculate Monte Carlo Returns (G_t)
        returns = []
        discounted_reward = 0
        for reward in reversed(memory.rewards):
            discounted_reward = reward + (gamma * discounted_reward)
            returns.insert(0, discounted_reward)
            
        returns = torch.FloatTensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Calculate Advantages (G_t - V(s))
        advantages = returns - old_values
        
        # 3. PPO Update (Train K_epochs on the same data!)
        for _ in range(K_epochs):
            # Evaluate the old actions using the NEW currently-updating policy
            new_log_probs, new_values = network.evaluate(old_states, old_actions)
            
            # ============================================================
            # TODO: Implement the PPO Clipped Surrogate Objective
            # 1. Calculate ratios (exp of new - old log probs)
            # 2. Calculate surr1 (ratios * advantages)
            # 3. Calculate surr2 (clamp ratios between 1-eps and 1+eps, mult by advantages)
            # 4. actor_loss = -torch.min(surr1, surr2).mean()
            #    Remember to `.detach()` the advantages!
            # ============================================================
            raise NotImplementedError
            
            # Standard Critic MSE Loss
            critic_loss = F.mse_loss(new_values.squeeze(), returns.detach())
            
            # PPO combined Loss
            loss = actor_loss + 0.5 * critic_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        # Clear memory after the K epochs are done to prepare for the next new rollout
        memory.clear()
        
        if (ep + 1) % 50 == 0:
            avg = np.mean(episode_rewards[-50:])
            print(f"Episode {ep+1:3d} | Avg Reward: {avg:.1f}")
            
    return network, episode_rewards

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

```python
ratios = torch.exp(new_log_probs - old_log_probs.detach())
surr1 = ratios * advantages.detach()
surr2 = torch.clamp(ratios, 1 - eps_clip, 1 + eps_clip) * advantages.detach()
actor_loss = -torch.min(surr1, surr2).mean()
```

</details>

In [ ]:
trained_net, ppo_rewards = train_ppo(num_episodes=500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ppo_rewards, color='#10b981', alpha=0.3)

window = 20
if len(ppo_rewards) >= window:
    smoothed = np.convolve(ppo_rewards, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(ppo_rewards)), smoothed, color='#10b981', linewidth=2, label=f'PPO (Moving Avg)')

ax.set_title("PPO Training on CartPole-v1")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.axhline(500, color='gray', linestyle='--', label="Max Reward")
ax.legend()
plt.show()

## 4. Watch the Agent Play!

Let's render our extremely stable PPO agent.

In [ ]:
def render_agent(env_name, agent):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset(seed=42)
    frames = []
    
    done = False
    truncated = False
    while not (done or truncated) and len(frames) < 500:
        frames.append(env.render())
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            action_probs, _ = agent(state_tensor)
            action = action_probs.argmax().item() # We can safely act greedily here!
        state, _, done, truncated, _ = env.step(action)
    env.close()
    
    fig, ax = plt.subplots(figsize=(6,4))
    ax.axis('off')
    img = ax.imshow(frames[0])
    
    def animate(i):
        img.set_data(frames[i])
        return [img]
        
    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
    plt.close() # Prevent extra static plot from showing
    return HTML(anim.to_jshtml())

render_agent('CartPole-v1', trained_net)


## Conclusion

Look at how beautiful and stable that learning curve is! Compare it to the massive, terrifying dips in the Actor-Critic plot we saw in Notebook 08.

By adding a single `torch.clamp()` mathematical operation, PPO completely prevents the massive, chaotic gradient norms that were shattering the network. 

Furthermore, because it uses the `RolloutBuffer` safely over multiple `K_epochs`, PPO is magnitudes more sample-efficient than REINFORCE. It squeezes every drop of juice out of its experience before moving on.

You have now implemented **Proximal Policy Optimization**. The exact same algorithmic math concept inside your `train_ppo` loop is the underlying engine that trained ChatGPT and DeepMind's dexterity robotic hands!

### What's Next?
We've spent the entire tutorial playing with a discrete environment (CartPole, where you can only press button `0` or button `1`).

But what if we want to drive a car and turn the steering wheel exactly `12.5` degrees? 

In **Notebook 10**, we step entirely out of discrete actions and learn how to use Neural Networks to output **Continuous Actions**. Welcome to the domain of true Robotics.